[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Protein-Function-Prediction/COMP3608ProteinFunctionPrediction/blob/main/notebooks/ProteinFunctionPrediction.ipynb)

In [ ]:
import sys, os

# ---------- Environment auto‑detection ----------
if 'google.colab' in sys.modules:
    # Running on Colab – clone the repo if not already present
    if not os.path.exists('COMP3608ProteinFunctionPrediction'):
        !git clone https://github.com/Protein-Function-Prediction/COMP3608ProteinFunctionPrediction.git
    os.chdir('COMP3608ProteinFunctionPrediction')
else:
    # Running locally (VSCode/Jupyter) – ensure we are at the project root
    if not os.path.exists('requirements.txt'):
        # The notebook is inside the 'notebooks' folder, go up one level
        os.chdir('..')
    # Final check
    if not os.path.exists('requirements.txt'):
        raise RuntimeError("Could not find the project root (requirements.txt). Run from the repo folder.")

!pip install -r requirements.txt

# **Protein Function Prediction – Data Preparation & Exploratory Data Analysis**
**Project:** COMP 3608 B-Rank Mission  
**Notebook:** Downloads Kaggle datasets → `data/raw/`, cleans & merges them,
engineers features, translates all labels to English, and saves processed data to `data/processed/`.
Rich EDA visualisations are generated to inform modelling decisions.

---


## **0. Environment Setup**

In [ ]:
import warnings, os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import kagglehub

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

# **Directory structure**
RAW_DIR   = Path('data/raw')
PROC_DIR  = Path('data/processed')
FIG_DIR   = Path('figures')
for d in [RAW_DIR / 'go_annotations', RAW_DIR / 'simulated_1',
          RAW_DIR / 'simulated_2', PROC_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)
print("✔ Environment ready. Directories created.")


## **1. Download Datasets from Kaggle**

In [ ]:
# GO annotations (real UniProt proteins)
path_go = kagglehub.dataset_download(
    "nikitamanaenkov/protein-sequences-with-go-annotations",
    output_dir=str(RAW_DIR / 'go_annotations')
)
# Simulated bioinformatics dataset 1
path_sim1 = kagglehub.dataset_download(
    "willianoliveiragibin/bioinformatics-simulated",
    output_dir=str(RAW_DIR / 'simulated_1')
)
# Simulated bioinformatics dataset 2
path_sim2 = kagglehub.dataset_download(
    "gallo33henrique/bioinformatics-protein-dataset-simulated",
    output_dir=str(RAW_DIR / 'simulated_2')
)
print("✔ All datasets downloaded to:", str(RAW_DIR))


## **2. Load & Inspect Raw Data**

In [ ]:
def find_csv(directory):
    return list(Path(directory).rglob('*.csv'))

df_go   = pd.read_csv(find_csv(path_go)[0])
df_sim1 = pd.read_csv(find_csv(path_sim1)[0])
df_sim2 = pd.read_csv(find_csv(path_sim2)[0])

print("── GO Annotations ──────────────────────────────────────")
print(f"  Shape : {df_go.shape}")
print(f"  Columns: {df_go.columns.tolist()}")
print()
print("── Simulated Dataset 1 ─────────────────────────────────")
print(f"  Shape : {df_sim1.shape}")
print(f"  Columns: {df_sim1.columns.tolist()}")
print()
print("── Simulated Dataset 2 ─────────────────────────────────")
print(f"  Shape : {df_sim2.shape}")
print(f"  Columns: {df_sim2.columns.tolist()}")


## **3. Standardise Columns to `sequence` and `label_raw`**

In [ ]:
def standardise_columns(df):
    """Auto-detect sequence and label columns, rename to standard names."""
    df = df.copy()
    seq_keywords   = ['seq', 'sequência', 'sequencia']
    label_keywords = ['go', 'label', 'function', 'class', 'classe']

    seq_col = next((c for c in df.columns
                    if any(k in c.lower() for k in seq_keywords)), None)
    if seq_col is None:
        raise KeyError(f"No sequence column found in {list(df.columns)}")

    label_col = next((c for c in df.columns
                      if any(k in c.lower() for k in label_keywords)), None)
    if label_col is None:
        raise KeyError(f"No label column found in {list(df.columns)}")

    print(f"  Detected → sequence='{seq_col}', label='{label_col}'")
    df.rename(columns={seq_col: 'sequence', label_col: 'label_raw'}, inplace=True)
    return df[['sequence', 'label_raw']]

print("── Column detection ────────────────────────────────────")
df_go   = standardise_columns(df_go)
df_sim1 = standardise_columns(df_sim1)
df_sim2 = standardise_columns(df_sim2)

print()
print("GO sample:")
display(df_go.head(3))


## **4. Label Mapping – All Labels to English**

### **4a. GO Annotation Labels → English Functional Categories**
Raw GO annotation strings (e.g. `['catalytic activity', 'ATP binding [GO:0005524]']`)
are mapped to one of six high-level **English** categories.


In [ ]:
# ── GO term → English category ────────────────────────────────────────────
def map_go_to_english(go_str):
    """Map a raw GO annotation string to an English functional category."""
    if pd.isna(go_str):
        return 'other'
    s = str(go_str).lower()
    if any(t in s for t in ['catalytic', 'hydrolase', 'kinase', 'transferase', 'enzyme']):
        return 'enzyme'
    if 'binding' in s or 'receptor' in s:
        return 'binding_receptor'
    if 'transporter' in s or 'transport' in s:
        return 'transporter'
    if 'signal' in s or 'transducer' in s:
        return 'signal_transduction'
    if 'structural' in s or 'cytoskeleton' in s:
        return 'structural'
    return 'other'

df_go['label'] = df_go['label_raw'].apply(map_go_to_english)
df_go.drop(columns='label_raw', inplace=True)
print("GO label distribution:")
print(df_go['label'].value_counts())


### **4b. Simulated Dataset Labels – Portuguese → English Translation**

In [ ]:
# Portuguese → English translation mapping
PT_TO_EN = {
    'enzima'        : 'enzyme',
    'estrutural'    : 'structural',
    'transporte'    : 'transporter',
    'receptora'     : 'binding_receptor',
    'outras'        : 'other',
    'sinalização'   : 'signal_transduction',
    'sinalizacao'   : 'signal_transduction',
    'catalitica'    : 'enzyme',
    'catalítica'    : 'enzyme',
}

def clean_and_translate(raw):
    """Normalise Portuguese label → English equivalent."""
    if pd.isna(raw):
        return 'other'
    label = str(raw).strip().lower().replace(' ', '_')
    # Direct translation lookup (try progressively shorter keys)
    for pt, en in PT_TO_EN.items():
        if pt in label:
            return en
    return label  # already English or unknown

for df in [df_sim1, df_sim2]:
    df['label'] = df['label_raw'].apply(clean_and_translate)
    df.drop(columns='label_raw', inplace=True)

print("Simulated 1 label distribution:")
print(df_sim1['label'].value_counts())
print()
print("Simulated 2 label distribution:")
print(df_sim2['label'].value_counts())


## **5. Merge, Validate & Clean Sequences**

In [ ]:
VALID_AA = set('ACDEFGHIKLMNPQRSTVWY')

def clean_sequence(s):
    """Replace invalid amino acid characters with 'X'; upper-case everything."""
    return ''.join(aa if aa in VALID_AA else 'X' for aa in str(s).upper())

df_all = pd.concat([df_go, df_sim1, df_sim2], ignore_index=True)
df_all.dropna(subset=['sequence', 'label'], inplace=True)
df_all['sequence'] = df_all['sequence'].apply(clean_sequence)

# Length filter (10–1 000 aa) and de-duplication
df_all = df_all[df_all['sequence'].str.len().between(10, 1000)]
df_all.drop_duplicates(subset='sequence', inplace=True)

# Remove classes with < 5 samples (too few to model)
vc = df_all['label'].value_counts()
df_all = df_all[df_all['label'].isin(vc[vc >= 5].index)].copy()

df_all['seq_len'] = df_all['sequence'].str.len()

print(f"Final dataset shape : {df_all.shape}")
print()
print("Label distribution (English):")
print(df_all['label'].value_counts())


## **6. Save Processed Data**

In [ ]:
# ── Save raw sequences for CNN notebook ─────────────────────────────────
df_all.to_csv(PROC_DIR / 'processed_sequences.csv', index=False)

# 95th-percentile sequence length (used for CNN padding)
max_len = int(np.percentile(df_all['seq_len'], 95))
np.save(PROC_DIR / 'max_len.npy', max_len)
print(f"Padding length (95th percentile): {max_len}")

# ── Feature engineering for classical & CNN models ──────────────────────
AA_ORDER  = 'ACDEFGHIKLMNPQRSTVWY'
AA_TO_IDX = {aa: i for i, aa in enumerate(AA_ORDER)}

def aac(seq):
    """Amino Acid Composition (20-dim frequency vector)."""
    counts = np.zeros(20)
    for aa in seq:
        if aa in AA_TO_IDX:
            counts[AA_TO_IDX[aa]] += 1
    return counts / len(seq) if len(seq) > 0 else counts

def dpc(seq):
    """Dipeptide Composition (400-dim frequency vector)."""
    vec = np.zeros(400)
    if len(seq) < 2:
        return vec
    for i in range(len(seq) - 1):
        a, b = seq[i], seq[i+1]
        if a in AA_TO_IDX and b in AA_TO_IDX:
            vec[AA_TO_IDX[a] * 20 + AA_TO_IDX[b]] += 1
    return vec / (len(seq) - 1)

X_aac = np.array([aac(s) for s in df_all['sequence']])
X_dpc = np.array([dpc(s) for s in df_all['sequence']])

le = LabelEncoder()
y  = le.fit_transform(df_all['label'])
class_names = le.classes_

np.savez(PROC_DIR / 'features.npz', X_aac=X_aac, X_dpc=X_dpc, y=y)
np.save(PROC_DIR / 'class_names.npy', class_names)
print(f"AAC features : {X_aac.shape}")
print(f"DPC features : {X_dpc.shape}")
print(f"Classes ({len(class_names)}): {list(class_names)}")


---
## **7. Exploratory Data Analysis (EDA)**

### **7a. Class Distribution**


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart – counts
label_counts = df_all['label'].value_counts()
colors = sns.color_palette('Set2', len(label_counts))
axes[0].bar(label_counts.index, label_counts.values, color=colors, edgecolor='white', linewidth=0.8)
axes[0].set_title('**Class Distribution (Count)**', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Functional Class', fontsize=11)
axes[0].set_ylabel('Number of Proteins', fontsize=11)
axes[0].tick_params(axis='x', rotation=35)
for bar, cnt in zip(axes[0].patches, label_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 str(cnt), ha='center', va='bottom', fontsize=9)

# Pie chart – proportions
axes[1].pie(label_counts.values, labels=label_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=140, pctdistance=0.85,
            wedgeprops=dict(width=0.6, edgecolor='white'))
axes[1].set_title('**Class Proportion**', fontweight='bold', fontsize=13)

plt.suptitle('**Protein Functional Class Distribution**', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_class_distribution.png', bbox_inches='tight', dpi=150)
plt.show()
print("✔ Saved: figures/eda_class_distribution.png")


### **7b. Sequence Length Distribution**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram overall
axes[0].hist(df_all['seq_len'], bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(df_all['seq_len'].median(), color='crimson', linestyle='--', linewidth=2,
                label=f"Median = {df_all['seq_len'].median():.0f}")
axes[0].axvline(max_len, color='orange', linestyle='--', linewidth=2,
                label=f"95th pct = {max_len}")
axes[0].set_title('**Sequence Length Distribution**', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Sequence Length (aa)', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].legend()

# Box plot per class (using seaborn for proper ordering)
label_order = df_all.groupby('label')['seq_len'].median().sort_values().index.tolist()
sns.boxplot(x='label', y='seq_len', data=df_all, order=label_order,
            palette='Set2', ax=axes[1])
axes[1].set_title('**Sequence Length by Functional Class**', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Functional Class', fontsize=11)
axes[1].set_ylabel('Sequence Length (aa)', fontsize=11)
axes[1].tick_params(axis='x', rotation=35)
plt.suptitle('')

plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_sequence_lengths.png', bbox_inches='tight', dpi=150)
plt.show()
print("✔ Saved: figures/eda_sequence_lengths.png")


### **7c. Amino Acid Composition Heatmap (Mean per Class)**

In [ ]:
aac_df = pd.DataFrame(X_aac, columns=list(AA_ORDER))
aac_df['label'] = df_all['label'].values
mean_aac = aac_df.groupby('label').mean()

plt.figure(figsize=(16, 5))
sns.heatmap(mean_aac, cmap='YlOrRd', linewidths=0.4, annot=True, fmt='.3f',
            annot_kws={'size': 8}, cbar_kws={'label': 'Mean Frequency'})
plt.title('**Mean Amino Acid Composition per Functional Class**', fontweight='bold', fontsize=14)
plt.xlabel('Amino Acid', fontsize=12)
plt.ylabel('Functional Class', fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_aa_composition_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()
print("✔ Saved: figures/eda_aa_composition_heatmap.png")


### **7d. Pairwise Amino Acid Frequency Correlation (AAC Features)**

In [ ]:
corr_matrix = aac_df[list(AA_ORDER)].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', center=0,
            square=True, linewidths=0.3, annot=False,
            cbar_kws={'label': 'Pearson r', 'shrink': 0.8})
plt.title('**Pairwise Amino Acid Frequency Correlation**', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_aa_correlation_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()
print("✔ Saved: figures/eda_aa_correlation_heatmap.png")


### **7e. Top-20 Dipeptide Frequencies per Class**

In [ ]:
AA_PAIRS = [a+b for a in AA_ORDER for b in AA_ORDER]
dpc_df = pd.DataFrame(X_dpc, columns=AA_PAIRS)
dpc_df['label'] = df_all['label'].values
mean_dpc = dpc_df.groupby('label').mean()

# Select top-20 highest-variance dipeptides
top20_di = dpc_df[AA_PAIRS].var().nlargest(20).index.tolist()

plt.figure(figsize=(16, 5))
sns.heatmap(mean_dpc[top20_di], cmap='magma_r', linewidths=0.3,
            cbar_kws={'label': 'Mean Dipeptide Frequency'})
plt.title('**Top-20 High-Variance Dipeptide Frequencies per Functional Class**',
          fontweight='bold', fontsize=13)
plt.xlabel('Dipeptide', fontsize=11)
plt.ylabel('Functional Class', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_dipeptide_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()
print("✔ Saved: figures/eda_dipeptide_heatmap.png")


### **7f. Additional Visual: Global Amino Acid Frequency Bar Chart**

In [ ]:
global_aac = aac_df[list(AA_ORDER)].mean().sort_values(ascending=False)
plt.figure(figsize=(10, 5))
sns.barplot(x=global_aac.index, y=global_aac.values, palette='viridis')
plt.title('**Global Mean Amino Acid Frequency**', fontweight='bold', fontsize=14)
plt.xlabel('Amino Acid', fontsize=12)
plt.ylabel('Mean Frequency', fontsize=12)
for i, val in enumerate(global_aac.values):
    plt.text(i, val + 0.001, f"{val:.3f}", ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_global_aa_frequency.png', bbox_inches='tight', dpi=150)
plt.show()
print("✔ Saved: figures/eda_global_aa_frequency.png")


### **7g. Additional Visual: Sequence Length Density per Class**

In [ ]:
plt.figure(figsize=(12, 5))
for cls in label_order:
    subset = df_all[df_all['label'] == cls]
    sns.kdeplot(subset['seq_len'], label=cls, linewidth=2)
plt.title('**Sequence Length Density per Functional Class**', fontweight='bold', fontsize=14)
plt.xlabel('Sequence Length (aa)', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.legend(title='Functional Class')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_seqlen_density.png', bbox_inches='tight', dpi=150)
plt.show()
print("✔ Saved: figures/eda_seqlen_density.png")


### **7h. Additional Visual: Clustermap of AAC Features by Class**

In [ ]:
plt.figure(figsize=(14, 6))
sns.clustermap(mean_aac, method='ward', cmap='YlGnBu',
               linewidths=0.5, figsize=(12, 6),
               cbar_kws={'label': 'Mean Frequency'})
plt.suptitle('**Hierarchical Clustering of Classes by AAC**', fontweight='bold', fontsize=14, y=1.05)
plt.savefig(FIG_DIR / 'eda_aac_clustermap.png', bbox_inches='tight', dpi=150)
plt.show()
print("✔ Saved: figures/eda_aac_clustermap.png")


### **7i. Dataset Summary Statistics**

In [ ]:
summary = pd.DataFrame({
    'Dataset'       : ['GO Annotations', 'Simulated 1', 'Simulated 2', 'Combined'],
    'Records'       : [len(df_go), len(df_sim1), len(df_sim2), len(df_all)],
    'Unique Classes': [df_go['label'].nunique(), df_sim1['label'].nunique(),
                       df_sim2['label'].nunique(), df_all['label'].nunique()],
    'Median Seq Len': [
        int(df_go.merge(df_all[['sequence','seq_len']], on='sequence', how='inner')['seq_len'].median())
        if 'seq_len' in df_all.columns else '—',
        '—', '—',
        int(df_all['seq_len'].median())
    ]
})
print("=" * 55)
print("           DATASET SUMMARY")
print("=" * 55)
display(summary)
print("=" * 55)
print()
print(f"Feature shapes saved:")
print(f"  AAC  : {X_aac.shape}  →  data/processed/features.npz (X_aac)")
print(f"  DPC  : {X_dpc.shape}  →  data/processed/features.npz (X_dpc)")
print(f"  y    : {y.shape}      →  data/processed/features.npz (y)")
print(f"  CNN pad length: {max_len}  →  data/processed/max_len.npy")


---
## **8. Conclusion**

**Data preparation is complete.** All processed artefacts are now in `data/processed/`:

| File | Contents |
|---|---|
| `processed_sequences.csv` | Clean sequences + English labels |
| `features.npz` | AAC (20-D), DPC (400-D) feature matrices + encoded labels |
| `class_names.npy` | English class name array |
| `max_len.npy` | 95th-percentile sequence length for CNN padding |

➡ **Next:** Run `classical_model.ipynb` for Logistic Regression & SVM, then `cnn_model.ipynb` for the 1D-CNN.
